# 🚗 TakarMobil: Pengembangan Model Machine Learning (Phase 4)
**Proyek Akhir Machine Learning - Semester 5 | Kelas B2 - Kelompok 15**  
*TakarMobil: Sistem Estimasi Rentang Harga & Analisis Faktor Penentu Harga Mobil Bekas*

---

### 🎯 Tujuan Tahap Pemodelan (Phase 4):
1. **Membangun Baseline Statistik:** Menetapkan acuan performa dasar (*lower-bound benchmark*) menggunakan estimasi median harga per merek.
2. **Melatih 4 Algoritma Regresi:**
   - **Model 1: Ridge Regression** (Linear Regularized - L2)
   - **Model 2: Decision Tree Regressor** (Non-linear Tree Model)
   - **Model 3: Random Forest Regressor** (Ensemble Bagging)
   - **Model 4: XGBoost Regressor** (Ensemble Gradient Boosting)
3. **Hyperparameter Tuning:** Menerapkan optimasi parameter dengan validasi silang **5-Fold `GroupKFold`** pada data training untuk menghindari overfitting dan data leakage.
4. **Penyimpanan Seluruh Model:** Mengekspor seluruh model terbaik yang telah di-fit ke dalam format `.joblib` di folder `models/` agar siap digunakan pada evaluasi mendalam dan aplikasi web Streamlit.


## 1. Import Library & Setup Lingkungan
Memuat pustaka algoritma regresi, cross-validation, metrik evaluasi, serta pipeline scikit-learn.


In [1]:
import os
import json
import joblib
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("Seluruh pustaka pemodelan berhasil dimuat!")


Seluruh pustaka pemodelan berhasil dimuat!


## 2. Pemuatan Dataset Hasil Preprocessing (Train & Test)
Memuat `train_data.csv` (13.453 baris) dan `test_data.csv` (3.346 baris) yang telah disiapkan pada Phase 3 menggunakan deteksi path dinamis.


In [2]:
candidate_train = [
    "data/processed/train_data.csv",
    "../data/processed/train_data.csv"
]
candidate_test = [
    "data/processed/test_data.csv",
    "../data/processed/test_data.csv"
]

train_path = next((p for p in candidate_train if os.path.exists(p)), None)
test_path = next((p for p in candidate_test if os.path.exists(p)), None)

if train_path is None or test_path is None:
    raise FileNotFoundError("Dataset train atau test tidak ditemukan!")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Data Training: {train_df.shape[0]} baris x {train_df.shape[1]} kolom")
print(f"Data Testing : {test_df.shape[0]} baris x {test_df.shape[1]} kolom")

# Mendefinisikan Fitur dan Target
num_cols = ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
cat_cols = ['Merek_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']
all_features = num_cols + cat_cols

X_train = train_df[all_features]
y_train = train_df['Log_Harga']
y_train_raw = train_df['Harga_Num']

X_test = test_df[all_features]
y_test = test_df['Log_Harga']
y_test_raw = test_df['Harga_Num']

groups_train = train_df['Merek_Model']

print(f"Fitur Numerik   : {num_cols}")
print(f"Fitur Kategorik : {cat_cols}")


Data Training: 13453 baris x 28 kolom
Data Testing : 3346 baris x 28 kolom
Fitur Numerik   : ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
Fitur Kategorik : ['Merek_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']


## 3. Baseline Statistik (Median per Merek)
Sebagai syarat akademis yang baik, kita menetapkan **Baseline Statistik** sebagai pembanding. Baseline ini memprediksi harga mobil hanya berdasarkan nilai median log-harga dari merek mobil yang bersangkutan pada data training.
Model Machine Learning yang baik wajib memberikan peningkatan performa yang signifikan di atas baseline ini.


In [3]:
# Hitung median log-harga per merek pada data training
brand_median_log = y_train.groupby(X_train['Merek_Clean']).median()
print("Median Log-Harga per Merek (Training Set):")
print(brand_median_log)

# Prediksi pada data testing
pred_baseline_log = X_test['Merek_Clean'].map(brand_median_log)
pred_baseline_raw = np.expm1(pred_baseline_log)

# Evaluasi Baseline (Skala Rupiah Asli)
baseline_mae = mean_absolute_error(y_test_raw, pred_baseline_raw)
baseline_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_baseline_raw))
baseline_r2 = r2_score(y_test_raw, pred_baseline_raw)

# Evaluasi Baseline (Skala Log)
baseline_log_mae = mean_absolute_error(y_test, pred_baseline_log)
baseline_log_rmse = np.sqrt(mean_squared_error(y_test, pred_baseline_log))
baseline_log_r2 = r2_score(y_test, pred_baseline_log)

print(f"\n=== HASIL EVALUASI BASELINE STATISTIK ===")
print(f"Skala Log    -> MAE: {baseline_log_mae:.4f} | RMSE: {baseline_log_rmse:.4f} | R2: {baseline_log_r2:.4f}")
print(f"Skala Rupiah -> MAE: Rp {baseline_mae:,.0f} | RMSE: Rp {baseline_rmse:,.0f} | R2: {baseline_r2:.4f}")


Median Log-Harga per Merek (Training Set):
Merek_Clean
Daihatsu         18.577685
Honda            19.041257
Mercedes-Benz    20.026111
Suzuki           18.812723
Toyota           19.123778
Name: Log_Harga, dtype: float64



=== HASIL EVALUASI BASELINE STATISTIK ===
Skala Log    -> MAE: 0.3212 | RMSE: 0.4028 | R2: 0.4414
Skala Rupiah -> MAE: Rp 83,000,717 | RMSE: Rp 162,056,942 | R2: 0.4404


## 4. Konfigurasi ColumnTransformer & Skema 5-Fold GroupKFold
Untuk menghindari data leakage, preprocessing dilakukan di dalam scikit-learn `Pipeline`.
- Fitur numerik distandarisasi dengan `StandardScaler`.
- Fitur kategorikal di-encode dengan `OneHotEncoder(handle_unknown='ignore')`.
- Skema validasi silang menggunakan **`GroupKFold(n_splits=5)`** berdasarkan kolom grup `Merek_Model`.


In [4]:
# Definisi Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

# Definisi 5-Fold GroupKFold
cv_group = GroupKFold(n_splits=5)
print("Preprocessor dan 5-Fold GroupKFold siap digunakan!")


Preprocessor dan 5-Fold GroupKFold siap digunakan!


## 5. Model 1: Ridge Regression (Linear Regularized - L2)
Ridge Regression menambahkan penalti L2 terhadap kuadrat bobot koefisien linier untuk mencegah multikolinearitas dan overfitting.
Kita mencari nilai optimal dari parameter regularisasi $\alpha$ (`alpha`).


In [5]:
start_time = time.time()

pipe_ridge = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

param_grid_ridge = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
}

grid_ridge = GridSearchCV(
    estimator=pipe_ridge,
    param_grid=param_grid_ridge,
    cv=cv_group,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train, groups=groups_train)
ridge_time = time.time() - start_time

best_ridge = grid_ridge.best_estimator_
print(f"Selesai dalam {ridge_time:.2f} detik")
print(f"Parameter Terbaik Ridge: {grid_ridge.best_params_}")
print(f"Best CV RMSE (Log Scale): {-grid_ridge.best_score_:.4f}")

# Evaluasi pada Data Test
pred_ridge_log = best_ridge.predict(X_test)
pred_ridge_raw = np.expm1(pred_ridge_log)

ridge_mae = mean_absolute_error(y_test_raw, pred_ridge_raw)
ridge_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_ridge_raw))
ridge_r2 = r2_score(y_test_raw, pred_ridge_raw)

print(f"Test Set -> MAE: Rp {ridge_mae:,.0f} | RMSE: Rp {ridge_rmse:,.0f} | R2: {ridge_r2:.4f}")


Selesai dalam 18.21 detik
Parameter Terbaik Ridge: {'model__alpha': 100.0}
Best CV RMSE (Log Scale): 0.4339
Test Set -> MAE: Rp 63,531,552 | RMSE: Rp 130,702,690 | R2: 0.6360


## 6. Model 2: Decision Tree Regressor
Decision Tree membagi ruang fitur secara bertingkat untuk menangkap hubungan non-linier antara usia, jarak tempuh, dan merek terhadap harga mobil.
Parameter yang dituning:
- `max_depth`: Kedalaman maksimum pohon untuk mengontrol kompleksitas.
- `min_samples_leaf`: Jumlah minimum sampel pada daun akhir pohon untuk mencegah overfitting.


In [6]:
start_time = time.time()

pipe_dt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

param_grid_dt = {
    'model__max_depth': [5, 8, 12, 16],
    'model__min_samples_leaf': [5, 10, 20]
}

grid_dt = GridSearchCV(
    estimator=pipe_dt,
    param_grid=param_grid_dt,
    cv=cv_group,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_dt.fit(X_train, y_train, groups=groups_train)
dt_time = time.time() - start_time

best_dt = grid_dt.best_estimator_
print(f"Selesai dalam {dt_time:.2f} detik")
print(f"Parameter Terbaik Decision Tree: {grid_dt.best_params_}")
print(f"Best CV RMSE (Log Scale): {-grid_dt.best_score_:.4f}")

# Evaluasi pada Data Test
pred_dt_log = best_dt.predict(X_test)
pred_dt_raw = np.expm1(pred_dt_log)

dt_mae = mean_absolute_error(y_test_raw, pred_dt_raw)
dt_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_dt_raw))
dt_r2 = r2_score(y_test_raw, pred_dt_raw)

print(f"Test Set -> MAE: Rp {dt_mae:,.0f} | RMSE: Rp {dt_rmse:,.0f} | R2: {dt_r2:.4f}")


Selesai dalam 2.56 detik
Parameter Terbaik Decision Tree: {'model__max_depth': 8, 'model__min_samples_leaf': 20}
Best CV RMSE (Log Scale): 0.4357
Test Set -> MAE: Rp 66,413,555 | RMSE: Rp 127,331,194 | R2: 0.6545


## 7. Model 3: Random Forest Regressor (Ensemble Bagging)
Random Forest menggabungkan banyak pohon keputusan (*ensemble bagging*) dengan subsampling acak fitur dan data, menghasilkan prediksi yang stabil dan variansi rendah.
Parameter yang dituning:
- `n_estimators`: Jumlah pohon di dalam forest.
- `max_depth`: Kedalaman maksimum tiap pohon.
- `min_samples_leaf`: Batas minimum sampel per daun.


In [7]:
start_time = time.time()

pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid_rf = {
    'model__n_estimators': [100, 150],
    'model__max_depth': [12, 16],
    'model__min_samples_leaf': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv_group,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train, groups=groups_train)
rf_time = time.time() - start_time

best_rf = grid_rf.best_estimator_
print(f"Selesai dalam {rf_time:.2f} detik")
print(f"Parameter Terbaik Random Forest: {grid_rf.best_params_}")
print(f"Best CV RMSE (Log Scale): {-grid_rf.best_score_:.4f}")

# Evaluasi pada Data Test
pred_rf_log = best_rf.predict(X_test)
pred_rf_raw = np.expm1(pred_rf_log)

rf_mae = mean_absolute_error(y_test_raw, pred_rf_raw)
rf_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_rf_raw))
rf_r2 = r2_score(y_test_raw, pred_rf_raw)

print(f"Test Set -> MAE: Rp {rf_mae:,.0f} | RMSE: Rp {rf_rmse:,.0f} | R2: {rf_r2:.4f}")


Selesai dalam 37.34 detik
Parameter Terbaik Random Forest: {'model__max_depth': 12, 'model__min_samples_leaf': 5, 'model__n_estimators': 150}
Best CV RMSE (Log Scale): 0.4500
Test Set -> MAE: Rp 66,832,338 | RMSE: Rp 126,624,895 | R2: 0.6583


## 8. Model 4: XGBoost Regressor (Gradient Boosting)
XGBoost (*Extreme Gradient Boosting*) membangun pohon keputusan secara sekuensial (*boosting*), di mana setiap pohon baru mengoreksi residual kesalahan dari pohon sebelumnya secara matematis efisien.
Parameter yang dituning:
- `n_estimators`: Jumlah tahapan boosting.
- `max_depth`: Kedalaman pohon.
- `learning_rate` ($\eta$): Laju penyusutan kontribusi tiap pohon.
- `subsample`: Rasio sampel acak tiap iterasi boosting.


In [8]:
start_time = time.time()

pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror'))
])

param_grid_xgb = {
    'model__n_estimators': [100, 150],
    'model__max_depth': [4, 6],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    cv=cv_group,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train, groups=groups_train)
xgb_time = time.time() - start_time

best_xgb = grid_xgb.best_estimator_
print(f"Selesai dalam {xgb_time:.2f} detik")
print(f"Parameter Terbaik XGBoost: {grid_xgb.best_params_}")
print(f"Best CV RMSE (Log Scale): {-grid_xgb.best_score_:.4f}")

# Evaluasi pada Data Test
pred_xgb_log = best_xgb.predict(X_test)
pred_xgb_raw = np.expm1(pred_xgb_log)

xgb_mae = mean_absolute_error(y_test_raw, pred_xgb_raw)
xgb_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_xgb_raw))
xgb_r2 = r2_score(y_test_raw, pred_xgb_raw)

print(f"Test Set -> MAE: Rp {xgb_mae:,.0f} | RMSE: Rp {xgb_rmse:,.0f} | R2: {xgb_r2:.4f}")


Selesai dalam 16.43 detik
Parameter Terbaik XGBoost: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 100, 'model__subsample': 1.0}
Best CV RMSE (Log Scale): 0.4255
Test Set -> MAE: Rp 64,077,415 | RMSE: Rp 119,971,501 | R2: 0.6933


## 9. Tabel Perbandingan Evaluasi Seluruh Model
Tabel ringkasan metrik evaluasi (MAE, MSE, RMSE, R²) pada data testing (skala nominal Rupiah asli).


In [9]:
results_data = {
    'Model': ['Baseline Statistik (Median)', 'Ridge Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'MAE (Rupiah)': [baseline_mae, ridge_mae, dt_mae, rf_mae, xgb_mae],
    'RMSE (Rupiah)': [baseline_rmse, ridge_rmse, dt_rmse, rf_rmse, xgb_rmse],
    'R² Score': [baseline_r2, ridge_r2, dt_r2, rf_r2, xgb_r2],
    'Training Time (s)': [0.01, ridge_time, dt_time, rf_time, xgb_time]
}

df_results = pd.DataFrame(results_data)
# Hitung MSE
df_results['MSE (Rupiah)'] = df_results['RMSE (Rupiah)'] ** 2

# Urutkan berdasarkan R² tertinggi
df_results = df_results.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

# Tampilkan Format Rapi
display_table = df_results.copy()
display_table['MAE (Rupiah)'] = display_table['MAE (Rupiah)'].apply(lambda x: f"Rp {x:,.0f}")
display_table['RMSE (Rupiah)'] = display_table['RMSE (Rupiah)'].apply(lambda x: f"Rp {x:,.0f}")
display_table['MSE (Rupiah)'] = display_table['MSE (Rupiah)'].apply(lambda x: f"{x:.2e}")
display_table['R² Score'] = display_table['R² Score'].apply(lambda x: f"{x:.4f}")
display_table['Training Time (s)'] = display_table['Training Time (s)'].apply(lambda x: f"{x:.2f} s")

print("=== TABEL PERBANDINGAN MODEL PADA TESTING SET ===")
display_table[['Model', 'MAE (Rupiah)', 'RMSE (Rupiah)', 'MSE (Rupiah)', 'R² Score', 'Training Time (s)']]


=== TABEL PERBANDINGAN MODEL PADA TESTING SET ===


,Model,MAE (Rupiah),RMSE (Rupiah),MSE (Rupiah),R² Score,Training Time (s)
0,XGBoost,"Rp 64,077,415","Rp 119,971,501",1.44e+16,0.6933,16.43 s
1,Random Forest,"Rp 66,832,338","Rp 126,624,895",1.60e+16,0.6583,37.34 s
2,Decision Tree,"Rp 66,413,555","Rp 127,331,194",1.62e+16,0.6545,2.56 s
3,Ridge Regression,"Rp 63,531,552","Rp 130,702,690",1.71e+16,0.6360,18.21 s
4,Baseline Statistik (Median),"Rp 83,000,717","Rp 162,056,942",2.63e+16,0.4404,0.01 s


## 10. Penyimpanan Model Terlatih (.joblib)
Menyimpan keempat model yang telah dioptimasi beserta berkas ringkasan pelatihan ke direktori `models/`.


In [10]:
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)

# 1. Simpan Model Pipeline Lengkap
joblib.dump(best_ridge, os.path.join(models_dir, 'ridge_model.joblib'))
joblib.dump(best_dt, os.path.join(models_dir, 'decision_tree_model.joblib'))
joblib.dump(best_rf, os.path.join(models_dir, 'random_forest_model.joblib'))
joblib.dump(best_xgb, os.path.join(models_dir, 'xgboost_model.joblib'))

print("Seluruh berkas model .joblib berhasil disimpan di folder models/:")
print("- ridge_model.joblib")
print("- decision_tree_model.joblib")
print("- random_forest_model.joblib")
print("- xgboost_model.joblib")

# 2. Simpan Ringkasan Hasil Training ke JSON
summary_dict = {
    'models_evaluated': ['Baseline', 'Ridge', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'best_model': df_results.iloc[0]['Model'],
    'best_r2': float(df_results.iloc[0]['R² Score']),
    'best_params': {
        'Ridge': grid_ridge.best_params_,
        'Decision Tree': grid_dt.best_params_,
        'Random Forest': grid_rf.best_params_,
        'XGBoost': grid_xgb.best_params_
    },
    'results': df_results.to_dict(orient='records')
}

with open(os.path.join(models_dir, 'training_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary_dict, f, indent=2)

print(f"Ringkasan pelatihan disimpan ke: {os.path.join(models_dir, 'training_summary.json')}")


Seluruh berkas model .joblib berhasil disimpan di folder models/:
- ridge_model.joblib
- decision_tree_model.joblib
- random_forest_model.joblib
- xgboost_model.joblib
Ringkasan pelatihan disimpan ke: ../models\training_summary.json
